In [15]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [41]:
a = "My name is sayed, and i want to have a extremely high depth understanding in AI DOMAIN"
print("Total chars: ", len(a))
print("Total words: ", len(a.split(" ")))

# t = tiktoken.get_encoding("r50k_base")
t = tiktoken.encoding_for_model("gpt-4o")
e = t.encode(a)
len(e), e

Total chars:  86
Total words:  17


(19,
 [5444,
  1308,
  382,
  2891,
  295,
  11,
  326,
  575,
  1682,
  316,
  679,
  261,
  11069,
  1932,
  13636,
  10335,
  306,
  20837,
  192120])

In [42]:
for token in e:
    print(token, "->", repr(t.decode([token])))

5444 -> 'My'
1308 -> ' name'
382 -> ' is'
2891 -> ' say'
295 -> 'ed'
11 -> ','
326 -> ' and'
575 -> ' i'
1682 -> ' want'
316 -> ' to'
679 -> ' have'
261 -> ' a'
11069 -> ' extremely'
1932 -> ' high'
13636 -> ' depth'
10335 -> ' understanding'
306 -> ' in'
20837 -> ' AI'
192120 -> ' DOMAIN'


In [43]:
class MyDataset(Dataset):
  def __init__(self, txt, tokenizer, max_length, stride):
    self.inputs = []
    self.outputs = []

    # Populate the inputs and outputs
    token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

    for i in range(0, len(token_ids)-max_length, stride):
      self.inputs.append(torch.tensor(token_ids[i:i+max_length]))
      self.outputs.append(torch.tensor(token_ids[i+1:i+max_length+1]))


  def __len__(self):
    return len(self.inputs)

  def __getitem__(self, idx):
    return self.inputs[idx], self.outputs[idx]

Now Prepare the Dataset

In [44]:
tokenizer = tiktoken.encoding_for_model("gpt-4o")

with open("/content/the-verdict.txt", "r") as f:
  txt = f.read()

In [45]:
dataset = MyDataset(txt, tokenizer, 4, 4)

In [46]:
for idx in range(10):
    i, o = dataset[idx]
    print(f"Input: {i.tolist()} ---> Output: {o.tolist()}")

Input: [40, 148954, 3324, 4525] ---> Output: [148954, 3324, 4525, 10874]
Input: [10874, 165003, 33750, 7542] ---> Output: [165003, 33750, 7542, 261]
Input: [261, 12424, 59245, 375] ---> Output: [12424, 59245, 375, 6460]
Input: [6460, 261, 1899, 19807] ---> Output: [261, 1899, 19807, 4951]
Input: [4951, 375, 786, 480] ---> Output: [375, 786, 480, 673]
Input: [673, 860, 2212, 19005] ---> Output: [860, 2212, 19005, 316]
Input: [316, 668, 316, 9598] ---> Output: [668, 316, 9598, 484]
Input: [484, 11, 306, 290] ---> Output: [11, 306, 290, 4679]
Input: [4679, 328, 1232, 40373] ---> Output: [328, 1232, 40373, 11]
Input: [11, 501, 1458, 22664] ---> Output: [501, 1458, 22664, 1232]


Now Create the Dataloader

In [47]:
# Create dataloader
dataloader = DataLoader(
  dataset,
  batch_size=1,
  shuffle=False,
  drop_last=True,
  num_workers=0
)

In [51]:
dataloader=iter(dataloader)

In [52]:
next(dataloader)

[tensor([[    40, 148954,   3324,   4525]]),
 tensor([[148954,   3324,   4525,  10874]])]

In [53]:
next(dataloader)

[tensor([[ 10874, 165003,  33750,   7542]]),
 tensor([[165003,  33750,   7542,    261]])]

In [54]:
next(dataloader)

[tensor([[  261, 12424, 59245,   375]]),
 tensor([[12424, 59245,   375,  6460]])]

In [55]:
a, b = next(dataloader)

In [56]:
a

tensor([[ 6460,   261,  1899, 19807]])

In [57]:
b

tensor([[  261,  1899, 19807,  4951]])